In [12]:
import sys
import os

sys.path.append(os.path.abspath(".."))

In [13]:
from src.preprocess import preprocess_pipeline

X_train, X_test, y_train, y_test = preprocess_pipeline("../data/raw/Customer-Churn.csv")

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (5634, 30)
X_test shape: (1409, 30)
y_train shape: (5634,)
y_test shape: (1409,)


/Users/rohanmahendra/Documents/Governance_Project/src/preprocess.py:25: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace({'Yes': 1, 'No': 0})


In [14]:
import numpy as np

print("NaNs in X_train:", np.isnan(X_train.to_numpy()).sum())
print("NaNs in X_test:", np.isnan(X_test.to_numpy()).sum())
print("Infs in X_train:", np.isinf(X_train.to_numpy()).sum())
print("Infs in X_test:", np.isinf(X_test.to_numpy()).sum())

print("Max value in X_train:", np.nanmax(X_train.to_numpy()))
print("Min value in X_train:", np.nanmin(X_train.to_numpy()))

NaNs in X_train: 0
NaNs in X_test: 0
Infs in X_train: 0
Infs in X_test: 0
Max value in X_train: 3.0130901058031325
Min value in X_train: -3.0130901058031325


In [16]:
from src.train_model import train_models

results = train_models(X_train, X_test, y_train, y_test)

print(results)

/Users/rohanmahendra/Documents/Governance_Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/rohanmahendra/Documents/Governance_Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/rohanmahendra/Documents/Governance_Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/rohanmahendra/Documents/Governance_Project/.venv/lib/python3.9/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/rohanmahendra/Documents/Governance_Project/.venv/lib/python3.9/site-packages/sklearn

{'logistic_regression': {'accuracy': 0.8069552874378992, 'precision': 0.6583850931677019, 'recall': 0.5668449197860963, 'f1_score': 0.6091954022988506}, 'random_forest': {'accuracy': 0.7920511000709723, 'precision': 0.6391752577319587, 'recall': 0.49732620320855614, 'f1_score': 0.5593984962406015}}


In [17]:
from src.fairness_checks import fairness_check
import pandas as pd
import joblib

# load original data
original_df = pd.read_csv("../data/raw/Customer-Churn.csv")

# load trained model
model = joblib.load("../models/logistic_model.pkl")

fairness_results = fairness_check(X_test, y_test, model, original_df)

print(fairness_results)

{'gender_churn_rate': {'Female': 0.22270742358078602, 'Male': 0.23407202216066483}, 'gender_gap': 0, 'senior_churn_rate': {0: 0.19123841617523168, 1: 0.42792792792792794}, 'senior_gap': 0.23668951175269626}


/Users/rohanmahendra/Documents/Governance_Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/rohanmahendra/Documents/Governance_Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/rohanmahendra/Documents/Governance_Project/.venv/lib/python3.9/site-packages/sklearn/utils/extmath.py:203: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


In [18]:
from src.policy_engine import policy_check

final_decision = policy_check(results, fairness_results)

print(final_decision)

{'score': 3, 'decisions': {'accuracy': 'PASS', 'recall': 'PASS', 'gender_fairness': 'PASS', 'senior_fairness': 'FAIL'}, 'final_status': 'CONDITIONALLY APPROVED'}


In [19]:
from src.data_quality import check_data_quality
import pandas as pd

df = pd.read_csv("../data/raw/Customer-Churn.csv")

quality_report = check_data_quality(df)

print(quality_report)

{'total_missing_values': 0, 'duplicate_rows': 0, 'num_rows': 7043, 'num_columns': 21, 'quality_score': 100, 'status': 'PASS'}


In [20]:
from src.metadata_manager import generate_metadata
import pandas as pd

df = pd.read_csv("../data/raw/Customer-Churn.csv")

metadata = generate_metadata(df)

print(metadata)

{'dataset_name': 'Customer-Churn', 'num_rows': 7043, 'num_columns': 21, 'columns': ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'MonthlyCharges', 'TotalCharges', 'Churn'], 'data_types': {'customerID': 'object', 'gender': 'object', 'SeniorCitizen': 'int64', 'Partner': 'object', 'Dependents': 'object', 'tenure': 'int64', 'PhoneService': 'object', 'MultipleLines': 'object', 'InternetService': 'object', 'OnlineSecurity': 'object', 'OnlineBackup': 'object', 'DeviceProtection': 'object', 'TechSupport': 'object', 'StreamingTV': 'object', 'StreamingMovies': 'object', 'Contract': 'object', 'PaperlessBilling': 'object', 'PaymentMethod': 'object', 'MonthlyCharges': 'float64', 'TotalCharges': 'object', 'Churn': 'object'}, 'target_column': 'Churn', 'owner': 'Rohan Ma

In [43]:
from src.lineage_tracker import log_step

log_step("ingestion", "Loaded raw dataset")
log_step("data_quality", "Checked missing values and duplicates")
log_step("preprocessing", "Cleaned and encoded data")
log_step("model_training", "Trained logistic and random forest models")
log_step("fairness_check", "Evaluated fairness across groups")
log_step("policy_engine", "Generated final governance decision")

{'step': 'policy_engine',
 'description': 'Generated final governance decision',
 'timestamp': '2026-04-10 15:52:39'}

In [46]:
import importlib
import src.explainability as exp

importlib.reload(exp)
print(dir(exp))
from src.explainability import explain_model_with_shap
import joblib

model = joblib.load("../models/logistic_model.pkl")

shap_explanation = explain_model_with_shap(model, X_train, X_test)

print(shap_explanation)

['BASE_DIR', 'Path', 'REPORTS_DIR', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__spec__', 'explain_model', 'explain_model_with_shap', 'json', 'pd', 'plt', 'shap']
{'top_features_by_mean_abs_shap': [{'feature': 'tenure', 'importance': 1.0823128874826136}, {'feature': 'InternetService_Fiber optic', 'importance': 0.7842926210681468}, {'feature': 'MonthlyCharges', 'importance': 0.7824021514579638}, {'feature': 'Contract_Two year', 'importance': 0.45791286822361293}, {'feature': 'TotalCharges', 'importance': 0.40742709012890427}, {'feature': 'StreamingMovies_Yes', 'importance': 0.25446169087936776}, {'feature': 'StreamingTV_Yes', 'importance': 0.2530857879834553}, {'feature': 'Contract_One year', 'importance': 0.24745258794175978}, {'feature': 'MultipleLines_Yes', 'importance': 0.21351368136001703}, {'feature': 'PaymentMethod_Electronic check', 'importance': 0.1757893675836345}]}


In [45]:
import shap
print(shap.__version__)

0.49.1
